In [5]:
from google.colab import drive
import glob, zipfile
import pandas as pd

드라이브 연결 (처음에 권한 허용 팝업 뜸)

In [6]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


드라이브의 zip 찾아서 압축 풀기

In [9]:
zip_path = glob.glob('/content/drive/MyDrive/dooribeon/*.zip')[0]
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/raw')

CSV 읽기

In [19]:
csv_path = glob.glob('/content/raw/*.csv')[0]
df = pd.read_csv(csv_path, encoding='cp949')
pd.options.display.float_format = '{:,.1f}'.format

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1084942 entries, 0 to 1084941
Data columns (total 23 columns):
 #   Column   Non-Null Count    Dtype  
---  ------   --------------    -----  
 0   접수년도     1084942 non-null  int64  
 1   자치구코드    1084937 non-null  float64
 2   자치구명     1084937 non-null  object 
 3   법정동코드    1084937 non-null  float64
 4   법정동명     1084937 non-null  object 
 5   지번구분코드   823652 non-null   float64
 6   지번구분     823652 non-null   object 
 7   본번       824149 non-null   float64
 8   부번       824149 non-null   float64
 9   층        823657 non-null   float64
 10  계약일      1084942 non-null  int64  
 11  전월세구분    1084941 non-null  object 
 12  임대면적     1084942 non-null  float64
 13  보증금(만원)  1084942 non-null  int64  
 14  임대료(만원)  1084942 non-null  int64  
 15  건물명      824147 non-null   object 
 16  건축년도     1063624 non-null  float64
 17  건물용도     1084942 non-null  object 
 18  계약기간     1055819 non-null  object 
 19  신규계약구분   1032114 non-null  object 
 20  갱신

In [12]:
cols = ['자치구명', '법정동명', '전월세구분', '건물용도',
        '임대면적', '층', '보증금(만원)', '임대료(만원)']
df2 = df[cols].copy()
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1084942 entries, 0 to 1084941
Data columns (total 8 columns):
 #   Column   Non-Null Count    Dtype  
---  ------   --------------    -----  
 0   자치구명     1084937 non-null  object 
 1   법정동명     1084937 non-null  object 
 2   전월세구분    1084941 non-null  object 
 3   건물용도     1084942 non-null  object 
 4   임대면적     1084942 non-null  float64
 5   층        823657 non-null   float64
 6   보증금(만원)  1084942 non-null  int64  
 7   임대료(만원)  1084942 non-null  int64  
dtypes: float64(2), int64(2), object(4)
memory usage: 66.2+ MB


In [16]:
print(df2['전월세구분'].value_counts())
print()
print(df2['건물용도'].value_counts())
print()
print(df2['층'].value_counts().sort_index().head(10))

전월세구분
월세    661696
전세    423245
Name: count, dtype: int64

건물용도
아파트      444983
단독다가구    261285
연립다세대    220378
오피스텔     158296
Name: count, dtype: int64

층
-4.0        5
-2.0       48
-1.0     6379
 1.0    37170
 2.0    91190
 3.0    96931
 4.0    86633
 5.0    71299
 6.0    50611
 7.0    43925
Name: count, dtype: int64


In [27]:
print('시작:', len(df2))

df2 = df2[df2['전월세구분'] == '월세']
print('월세만:', len(df2))

df2 = df2[df2['건물용도'] != '아파트']
print('아파트 제외:', len(df2))

df2 = df2[~(df2['층'] < -1)]
print('지하 2층 이하 제외(널값은 포함):', len(df2))

시작: 466224
월세만: 466224
아파트 제외: 466224
지하 2층 이하 제외(널값은 포함): 466224


In [22]:
print(df2[['임대면적', '보증금(만원)', '임대료(만원)']].describe())


           임대면적   보증금(만원)   임대료(만원)
count 466,224.0 466,224.0 466,224.0
mean       34.3   5,140.7      57.5
std        25.3   7,536.7      43.4
min         5.8       0.0       0.0
25%        20.0   1,000.0      35.0
50%        28.0   2,000.0      52.0
75%        40.0   5,000.0      70.0
max       445.1 350,000.0   2,800.0


In [28]:
zero_deposit = df2[df2['보증금(만원)'] == 0]

print('보증금 0원:', len(zero_deposit), '건')
print('그중 월세도 0원:', (zero_deposit['임대료(만원)'] == 0).sum(), '건\n')

print(zero_deposit['건물용도'].value_counts(), '\n')
print(zero_deposit[['임대면적', '임대료(만원)']].describe(), '\n')

zero_deposit.head(10)

보증금 0원: 1449 건
그중 월세도 0원: 0 건

건물용도
단독다가구    866
연립다세대    583
Name: count, dtype: int64 

         임대면적  임대료(만원)
count 1,449.0  1,449.0
mean     53.7    115.9
std      52.2    194.1
min      10.0      5.0
25%      21.6     37.0
50%      33.0     50.0
75%      60.1     90.0
max     303.8  1,620.0 



,자치구명,법정동명,전월세구분,건물용도,임대면적,층,보증금(만원),임대료(만원)
1262,도봉구,쌍문동,월세,단독다가구,19.8,NaN,0,45
6036,용산구,이태원동,월세,단독다가구,99.2,NaN,0,238
6579,관악구,봉천동,월세,연립다세대,85.8,1.0,0,20
7209,용산구,신계동,월세,단독다가구,20.0,NaN,0,100
7295,서대문구,홍은동,월세,단독다가구,40.6,NaN,0,35
7618,강서구,화곡동,월세,연립다세대,61.7,-1.0,0,85
8408,영등포구,신길동,월세,단독다가구,20.1,NaN,0,31
8532,용산구,이태원동,월세,단독다가구,45.0,NaN,0,90
8936,금천구,시흥동,월세,단독다가구,166.3,NaN,0,20
9788,성북구,정릉동,월세,연립다세대,30.0,6.0,0,35


In [29]:
df2 = df2[df2['임대료(만원)'] > 0]
print('월세 0원 제외:', len(df2))

월세 0원 제외: 466187


In [30]:
df2 = df2[df2['임대면적'] <= 85]
print('면적 85㎡ 이하:', len(df2))

print(df2[['보증금(만원)', '임대료(만원)']].quantile([0.01, 0.05, 0.5, 0.95, 0.99]))

면적 85㎡ 이하: 453843
     보증금(만원)  임대료(만원)
0.0    100.0      4.0
0.1    300.0     10.0
0.5  2,000.0     52.0
0.9 21,000.0    115.0
1.0 30,000.0    171.0


In [31]:
print(df2[['임대면적', '보증금(만원)', '임대료(만원)']].describe())

           임대면적   보증금(만원)   임대료(만원)
count 453,843.0 453,843.0 453,843.0
mean       31.2   5,062.2      56.4
std        14.8   7,162.7      36.3
min         5.8       0.0       1.0
25%        19.9   1,000.0      35.0
50%        27.4   2,000.0      52.0
75%        39.2   5,000.0      70.0
max        85.0 294,000.0   1,300.0


In [32]:
df2.sort_values('임대료(만원)', ascending=False).head(15)

,자치구명,법정동명,전월세구분,건물용도,임대면적,층,보증금(만원),임대료(만원)
977776,서초구,방배동,월세,연립다세대,49.3,4.0,3000,1300
975809,서초구,방배동,월세,연립다세대,49.3,4.0,3000,1300
1072756,마포구,마포동,월세,연립다세대,71.3,4.0,3000,1200
1072778,마포구,마포동,월세,연립다세대,53.5,4.0,3000,1200
1072752,마포구,마포동,월세,연립다세대,57.8,4.0,3000,1200
1072777,마포구,마포동,월세,연립다세대,53.5,4.0,3000,1200
1072796,마포구,마포동,월세,연립다세대,59.7,4.0,3000,1200
1072751,마포구,마포동,월세,연립다세대,57.8,4.0,3000,1200
1072795,마포구,마포동,월세,연립다세대,59.7,4.0,3000,1200
329850,마포구,마포동,월세,연립다세대,59.9,4.0,3000,1200


In [33]:
over = df2[df2['임대료(만원)'] >= 1000]

print('월세 1000만 원 이상:', len(over), '건\n')
over.groupby(['자치구명', '법정동명', '보증금(만원)', '임대료(만원)']).size()

월세 1000만 원 이상: 25 건



,,,,0
자치구명,법정동명,보증금(만원),임대료(만원),
광진구,구의동,3000,1100,1
마포구,마포동,3000,1200,22
서초구,방배동,3000,1300,2


In [34]:
bins = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, float('inf')]
labels = ['100만 미만', '100만대', '200만대', '300만대', '400만대', '500만대',
          '600만대', '700만대', '800만대', '900만대', '1000만 이상']

rent_range = pd.cut(df2['임대료(만원)'], bins=bins, labels=labels, right=False)
rent_range.value_counts().sort_index()

,count
임대료(만원),
100만 미만,410663
100만대,40277
200만대,2466
300만대,276
400만대,54
500만대,22
600만대,42
700만대,7
800만대,0


In [36]:
df2 = df2[df2['임대료(만원)'] < 400]
print('월세 400만 원 이상 제외:', len(df2))

월세 400만 원 이상 제외: 453682


In [37]:
df2.isna().sum()

,0
자치구명,0
법정동명,0
전월세구분,0
건물용도,0
임대면적,0
층,205263
보증금(만원),0
임대료(만원),0


In [38]:
counts = df2.groupby(['자치구명', '법정동명']).size().sort_values(ascending=False)

print('전체 동네 수:', len(counts))
print()
print(counts.describe())
counts.head(30)

전체 동네 수: 394

count      394.0
mean     1,151.5
std      2,108.8
min          1.0
25%         70.5
50%        350.0
75%      1,387.2
max     19,687.0
dtype: float64


자치구명  법정동명
관악구   봉천동     19687
      신림동     18788
강서구   화곡동     12601
      마곡동      9639
송파구   문정동      7743
금천구   독산동      7603
구로구   구로동      7057
강남구   역삼동      6850
강동구   천호동      6651
광진구   자양동      6589
동작구   사당동      6554
광진구   화양동      6341
중랑구   면목동      6317
광진구   중곡동      6182
동작구   상도동      5939
광진구   구의동      5841
송파구   방이동      5731
마포구   성산동      5300
강북구   수유동      5160
강동구   성내동      4745
송파구   석촌동      4604
서초구   서초동      4573
양천구   목동       4420
송파구   삼전동      4321
      잠실동      4291
강서구   등촌동      4249
노원구   공릉동      4225
서대문구  연희동      4219
서초구   방배동      4190
동대문구  휘경동      4088
dtype: int64

In [40]:
counts = df2.groupby(['자치구명', '법정동명']).size().sort_values(ascending=False)

print('전체 동네 수:', len(counts))
print(counts.describe())
counts.tail(10)

전체 동네 수: 394
count      394.0
mean     1,151.5
std      2,108.8
min          1.0
25%         70.5
50%        350.0
75%      1,387.2
max     19,687.0
dtype: float64


자치구명  법정동명 
중구    을지로7가    3
      을지로2가    2
종로구   화동       2
서초구   원지동      2
종로구   내자동      2
      소격동      2
강남구   압구정동     2
영등포구  양평동      1
종로구   묘동       1
용산구   용산동1가    1
dtype: int64

In [41]:
import requests
from google.colab import userdata

KAKAO_KEY = userdata.get('KAKAO_REST_KEY')

def get_coord(gu, dong):
    url = 'https://dapi.kakao.com/v2/local/search/address.json'
    headers = {'Authorization': f'KakaoAK {KAKAO_KEY}'}
    res = requests.get(url, headers=headers, params={'query': f'서울특별시 {gu} {dong}'})
    res.raise_for_status()
    docs = res.json()['documents']
    if not docs:
        return None, None
    return float(docs[0]['y']), float(docs[0]['x'])   # y = 위도, x = 경도

get_coord('마포구', '서교동')
# (37.55..., 126.92...)

(37.5550949138083, 126.914613374473)

In [43]:
import os, time

save_path = '/content/drive/MyDrive/dooribeon/neighborhood_coords.csv'

if os.path.exists(save_path):
    # 이미 저장된 파일이 있으면 API 안 부르고 읽기만
    nb = pd.read_csv(save_path)
    print('저장된 파일에서 불러옴:', len(nb), '개')
else:
    # 동네 목록 만들기 (구, 동, 계약 건수)
    nb = counts.reset_index(name='contract_count')
    nb.columns = ['gu', 'dong', 'contract_count']

    lats, lngs = [], []
    for i, row in nb.iterrows():
        lat, lng = get_coord(row['gu'], row['dong'])
        lats.append(lat)
        lngs.append(lng)
        time.sleep(0.05)                      # 너무 빠르게 연속 호출하지 않게 잠깐 쉬기
        if (i + 1) % 50 == 0:
            print(f'{i + 1}/{len(nb)} 완료')

    nb['lat'] = lats
    nb['lng'] = lngs
    nb.to_csv(save_path, index=False, encoding='utf-8-sig')
    print('드라이브에 저장 완료:', save_path)

print('좌표 못 찾은 동네:', nb['lat'].isna().sum(), '개')
nb.head()

50/394 완료
100/394 완료
150/394 완료
200/394 완료
250/394 완료
300/394 완료
350/394 완료
드라이브에 저장 완료: /content/drive/MyDrive/dooribeon/neighborhood_coords.csv
좌표 못 찾은 동네: 0 개


,gu,dong,contract_count,lat,lng
0,관악구,봉천동,19687,37.5,127.0
1,관악구,신림동,18788,37.5,126.9
2,강서구,화곡동,12601,37.5,126.8
3,강서구,마곡동,9639,37.6,126.8
4,송파구,문정동,7743,37.5,127.1


In [44]:
pd.reset_option('display.float_format')   # 숫자 표시 설정 원래대로
nb.head()

,gu,dong,contract_count,lat,lng
0,관악구,봉천동,19687,37.482538,126.952165
1,관악구,신림동,18788,37.489452,126.926185
2,강서구,화곡동,12601,37.536285,126.848952
3,강서구,마곡동,9639,37.574786,126.830205
4,송파구,문정동,7743,37.484742,127.118268


In [45]:
print(nb[['lat', 'lng']].describe())

out = nb[(nb['lat'] < 37.4) | (nb['lat'] > 37.7) | (nb['lng'] < 126.7) | (nb['lng'] > 127.2)]
print('서울 범위 밖:', len(out), '개')
out

              lat         lng
count  394.000000  394.000000
mean    37.555354  126.984562
std      0.039083    0.068751
min     37.443312  126.810109
25%     37.532137  126.943509
50%     37.561265  126.984682
75%     37.580133  127.021945
max     37.687396  127.173906
서울 범위 밖: 0 개


,gu,dong,contract_count,lat,lng


In [46]:
rent_contracts = df2.rename(columns={
    '자치구명': 'gu',
    '법정동명': 'dong',
    '건물용도': 'building_type',
    '임대면적': 'area',
    '층': 'floor',
    '보증금(만원)': 'deposit',
    '임대료(만원)': 'rent',
})[['gu', 'dong', 'building_type', 'area', 'floor', 'deposit', 'rent']]

rent_contracts_path = '/content/drive/MyDrive/dooribeon/rent_contracts.csv'
rent_contracts.to_csv(rent_contracts_path, index=False, encoding='utf-8-sig')

print('계약 건수:', len(rent_contracts))
print('파일 크기:', round(os.path.getsize(rent_contracts_path) / 1024 / 1024, 1), 'MB')
rent_contracts.head()

계약 건수: 453682
파일 크기: 22.6 MB


,gu,dong,building_type,area,floor,deposit,rent
16,광진구,구의동,오피스텔,20.06,4.0,1000,60
17,영등포구,당산동1가,연립다세대,28.83,5.0,19500,30
19,금천구,독산동,연립다세대,29.90,3.0,50,75
20,송파구,방이동,연립다세대,45.90,2.0,20000,45
24,강동구,천호동,오피스텔,32.57,8.0,1000,82


In [47]:
rent_contracts['floor'] = rent_contracts['floor'].astype('Int64')   # 4.0 → 4, 빈 값은 그대로 유지

gz_path = '/content/drive/MyDrive/dooribeon/rent_contracts.csv.gz'
rent_contracts.to_csv(gz_path, index=False, encoding='utf-8', compression='gzip')

print('압축 파일 크기:', round(os.path.getsize(gz_path) / 1024 / 1024, 1), 'MB')

압축 파일 크기: 3.3 MB
